# Fine-tune EDM CIFAR-10 → CIFAR-100

Fine-tunes NVIDIA's pretrained CIFAR-10 EDM diffusion model on CIFAR-100, following the [MimicDiffusion](https://github.com/psky1111/MimicDiffusion) recipe: fine-tune the **unconditional** EDM checkpoint. This avoids the class-count mismatch you'd get fine-tuning the conditional checkpoint (10-way label embedding vs. CIFAR-100's 100 classes).

**Before running:** `Runtime` → `Change runtime type` → select a GPU (T4 or better).

Source of truth for this workflow: [`finetune_edm_cifar100.py`](./finetune_edm_cifar100.py) in this repo. If you hit an issue running this notebook, report it back in the Claude Code session that generated it -- fixes land in that `.py` file (and this notebook) and get pushed to this branch.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os
import torchvision

# ── Configuration ──
USE_DRIVE = False  # set True to persist repo/dataset/checkpoints/runs to Google Drive
drive_base_path = '/content/drive/MyDrive/edm-cifar100'  # only used if USE_DRIVE
# Paths below are quoted before being passed to shell magics, so spaces are
# tolerated -- but note this path is unrelated to where the notebook file
# itself lives; it's just where this script writes its own working files.

BASE = drive_base_path if USE_DRIVE else '/content'
os.makedirs(BASE, exist_ok=True)

COND = False  # unconditional fine-tuning (recommended -- see markdown cell above).
# COND=True is possible but the label-embedding weights won't transfer
# (shape mismatch: 10 classes -> 100), so --transfer effectively
# reinitializes that layer at random; only the backbone benefits.

DURATION_MIMG = 10  # fine-tuning budget, in millions of images
                    # (the paper's from-scratch CIFAR-10 run used 200; fine-tuning needs far less)
BATCH = 128         # lower than the paper's default of 512 to fit a single Colab GPU;
                    # scale --tick/--snap in the training cell if you raise this a lot

In [ ]:
# Only needed if USE_DRIVE=True above
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
# 1. Clone the NVlabs/edm repository
edm_dir = os.path.join(BASE, 'edm')
if not os.path.isdir(edm_dir):
    !git clone https://github.com/NVlabs/edm.git "{edm_dir}"

%cd "{edm_dir}"

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Prepare the CIFAR-100 dataset.
# dataset_tool.py only derives labels from a dataset.json or from top-level
# subfolder names -- NOT from filenames -- so save each image into a
# per-class subfolder (train/<label>/*.png) rather than one flat folder.
raw_dir = os.path.join(BASE, 'datasets', 'cifar100-raw', 'train')
os.makedirs(raw_dir, exist_ok=True)

trainset = torchvision.datasets.CIFAR100(
    root=os.path.join(BASE, 'data_temp'), train=True, download=True,
)
for i, (img, label) in enumerate(trainset):
    class_dir = os.path.join(raw_dir, f'{label:03d}')
    os.makedirs(class_dir, exist_ok=True)
    img.save(os.path.join(class_dir, f'{i:05d}.png'))
print(f"CIFAR-100 training images saved to {raw_dir}")

In [ ]:
# Convert to the zip format train.py expects (also picks up per-folder labels)
dataset_zip = os.path.join(BASE, 'datasets', 'cifar100-32x32.zip')
!python dataset_tool.py --source="{raw_dir}" --dest="{dataset_zip}" --resolution=32x32

In [ ]:
# 4. Download the pretrained EDM CIFAR-10 checkpoint (NVIDIA ships .pkl, not .pt)
checkpoints_dir = os.path.join(BASE, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)
ckpt_name = 'edm-cifar10-32x32-cond-vp.pkl' if COND else 'edm-cifar10-32x32-uncond-vp.pkl'
ckpt_path = os.path.join(checkpoints_dir, ckpt_name)
!wget -nc https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/{ckpt_name} -P "{checkpoints_dir}"

In [ ]:
# 5. Fine-tune.
# --transfer loads pretrained network weights for fine-tuning (tolerant of
# shape mismatches). --duration is in millions of images, not kimg.
# --tick/--snap are scaled down from the (50, 50) defaults so a short
# fine-tuning run still produces a handful of checkpoints, not just one
# at the very end.
outdir = os.path.join(BASE, 'training-runs-cifar100')
train_cmd = (
    f'python train.py '
    f'--outdir="{outdir}" '
    f'--data="{dataset_zip}" '
    f'--cond={"1" if COND else "0"} '
    f'--transfer="{ckpt_path}" '
    f'--duration={DURATION_MIMG} '
    f'--batch={BATCH} '
    f'--tick=10 '
    f'--snap=10 '
    f'--dump=10 '
    f'--metrics=none'
)
print(train_cmd)
!{train_cmd}

In [ ]:
print("Fine-tuning finished. Network snapshots (.pkl) are under:")
print(f"  {outdir}")